# Validación breve de entrenamiento con GPU en TensorFlow

Adaptación didáctica del [tutorial inicial de TensorFlow con MNIST](https://www.tensorflow.org/tutorials/quickstart/beginner) y de su [guía de GPU](https://www.tensorflow.org/guide/gpu). Usamos un subconjunto de dígitos para comprobar el entorno, no para conseguir un resultado de referencia.

Abre esta libreta con un **kernel nuevo de .venv**, desde `uv run jupyter lab`, después de terminar `uv sync`. Ejecuta de arriba abajo. No instala dependencias ni controladores; MNIST se descarga una vez (unos 11 MB) y queda en la caché local del proyecto.

El diagnóstico distingue:
1. GPU visible.
2. Operación matricial ejecutada en GPU.
3. Gradientes y actualización de pesos en GPU.
4. Entrenamiento y evaluación de un clasificador pequeño.
5. Comprobación independiente de LSTM nativa y cuDNN con datos sintéticos.

Una GPU detectada no demuestra por sí sola que se entrene en ella. Esta libreta comprueba dispositivos y cambios de pesos; no promete una aceleración frente a CPU.

### 1 · Configurar TensorFlow antes de usar la GPU

Fijamos rutas locales, semillas y crecimiento de memoria. Si TensorFlow ya se inicializó, reinicia el kernel para que se apliquen estas opciones. La libreta se detiene si no detecta GPU, en lugar de continuar silenciosamente en CPU.

In [1]:
import os
from pathlib import Path
import json, time, platform, subprocess
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    raise RuntimeError("Abre Jupyter desde la raíz de hotel_review.")
os.environ["KERAS_HOME"] = str(ROOT / ".cache" / "keras")
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import numpy as np
import tensorflow as tf
from tensorflow import keras
from IPython.display import display, Markdown

SEED = 42
gpus = tf.config.list_physical_devices("GPU")
if not gpus:
    raise RuntimeError("No hay GPU visible para TensorFlow. Revisa tensorflow[and-cuda], el controlador NVIDIA y WSL2.")
try:
    tf.config.threading.set_intra_op_parallelism_threads(4)
    tf.config.threading.set_inter_op_parallelism_threads(2)
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
except RuntimeError as error:
    raise RuntimeError("Reinicia el kernel y ejecuta esta celda primero.") from error
keras.utils.set_random_seed(SEED)
keras.mixed_precision.set_global_policy("float32")
REPORT_DIR = ROOT / "artifacts" / "gpu_validation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
gpu_info = [tf.config.experimental.get_device_details(gpu) for gpu in gpus]
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("Python:", platform.python_version())
display(gpu_info)


TensorFlow: 2.20.0 | Keras: 3.15.1
Python: 3.12.3


[{'compute_capability': (8, 6), 'device_name': 'NVIDIA GeForce RTX 3070 Ti'}]

### 2 · Ejecutar una operación real en GPU

Desactivamos temporalmente la colocación automática en otro dispositivo y hacemos una multiplicación de matrices. Consultamos el dispositivo del resultado y lo sincronizamos. Después restauramos la política para permitir que tareas auxiliares del entrenamiento se ejecuten en CPU cuando corresponda.

In [2]:
old_soft_placement = tf.config.get_soft_device_placement()
try:
    tf.config.set_soft_device_placement(False)
    with tf.device("/GPU:0"):
        product = tf.matmul(tf.ones((128, 128)), tf.ones((128, 128)))
    np.testing.assert_allclose(product.numpy(), 128.0)
    assert "device:GPU:0" in product.device
    print("Operación ejecutada en:", product.device)
finally:
    tf.config.set_soft_device_placement(old_soft_placement)


Operación ejecutada en: /job:localhost/replica:0/task:0/device:GPU:0


I0000 00:00:1788908438.898309   80415 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5555 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070 Ti, pci bus id: 0000:07:00.0, compute capability: 8.6


### 3 · Descargar MNIST y preparar un subconjunto

MNIST permite comprobar el entrenamiento con una descarga pequeña, independiente de Word2Vec. Normalizamos píxeles a [0,1] y reservamos 1,000 ejemplos de prueba. Usamos 6,000 ejemplos de entrenamiento y dos épocas para mantener breve el diagnóstico.

In [3]:
(x_train_all, y_train_all), (x_test_all, y_test_all) = keras.datasets.mnist.load_data()
rng = np.random.default_rng(SEED)
selected = rng.permutation(len(x_train_all))[:6000]
x_train = x_train_all[selected].astype("float32") / 255.0
y_train = y_train_all[selected].astype("int32")
x_test = x_test_all[:1000].astype("float32") / 255.0
y_test = y_test_all[:1000].astype("int32")
del x_train_all, y_train_all, x_test_all, y_test_all
print("Entrenamiento:", x_train.shape, "| Prueba:", x_test.shape)
print("MNIST local:", Path(os.environ["KERAS_HOME"]) / "datasets" / "mnist.npz")


Entrenamiento: (6000, 28, 28) | Prueba: (1000, 28, 28)
MNIST local: /home/maxkaizo/hotel_review/.cache/keras/datasets/mnist.npz


### 4 · Crear el clasificador y comprobar un paso de aprendizaje

Siguiendo la idea del tutorial, usamos Flatten, Dense, Dropout y una salida de diez logits. Inspeccionamos la ubicación de variables, logits y gradientes de un paso real, aplicamos el optimizador y comprobamos que los pesos cambien. Las tareas de coordinación o lectura de datos pueden seguir usando CPU.

In [4]:
with tf.device("/GPU:0"):
    model = keras.Sequential([
        keras.Input(shape=(28, 28)),
        keras.layers.Flatten(),
        keras.layers.Dense(128, activation="relu"),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10),
    ])
    optimizer = keras.optimizers.Adam()
    loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
    before_step = [variable.numpy().copy() for variable in model.trainable_variables]
    with tf.GradientTape() as tape:
        logits = model(tf.convert_to_tensor(x_train[:64]), training=True)
        loss = loss_fn(tf.convert_to_tensor(y_train[:64]), logits)
    gradients = tape.gradient(loss, model.trainable_variables)
    assert all(gradient is not None for gradient in gradients)
    assert "device:GPU:0" in logits.device
    gradient_devices = [gradient.device for gradient in gradients]
    variable_devices = [variable.value.device for variable in model.trainable_variables]
    assert all("device:GPU:0" in device for device in gradient_devices + variable_devices)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
assert any(not np.array_equal(before, after.numpy())
           for before, after in zip(before_step, model.trainable_variables))
assert np.isfinite(float(loss.numpy()))
print("Dispositivos de variables:", variable_devices)
print("Dispositivos de gradientes:", gradient_devices)
print("Pérdida del primer paso:", float(loss.numpy()))
print("Actualización de pesos verificada.")


Dispositivos de variables: ['/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0']
Dispositivos de gradientes: ['/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0']
Pérdida del primer paso: 2.3043019771575928
Actualización de pesos verificada.


### 5 · Entrenar durante dos épocas

Entrenamos después del paso de diagnóstico anterior. Usamos lotes pequeños y controlamos el número de hilos del lector de datos. El tiempo incluye preparación de la función y validación; no es un benchmark de velocidad GPU/CPU.

In [5]:
def batches(X, y, training=False):
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        dataset = dataset.shuffle(len(X), seed=SEED)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 2
    return dataset.batch(64).with_options(options).prefetch(1)

with tf.device("/GPU:0"):
    model.compile(optimizer=optimizer, loss=loss_fn,
                  metrics=["accuracy"], jit_compile=False)
    started = time.perf_counter()
    history = model.fit(batches(x_train[:5400], y_train[:5400], True),
                        validation_data=batches(x_train[5400:], y_train[5400:]),
                        epochs=2, verbose=2)
    training_seconds = time.perf_counter() - started
assert all(np.isfinite(values).all() for values in map(np.asarray, history.history.values()))
print("Tiempo de fit:", round(training_seconds, 2), "segundos")


Epoch 1/2


/home/maxkaizo/hotel_review/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


85/85 - 2s - 20ms/step - accuracy: 0.7450 - loss: 0.9280 - val_accuracy: 0.8750 - val_loss: 0.4580
Epoch 2/2
85/85 - 1s - 10ms/step - accuracy: 0.8778 - loss: 0.4227 - val_accuracy: 0.8983 - val_loss: 0.3634
Tiempo de fit: 3.58 segundos


### 6 · Evaluar y guardar evidencia del entrenamiento

Evaluamos el subconjunto reservado y registramos resultados reales. No imponemos un umbral arbitrario de exactitud para decidir si la GPU funciona: la evidencia clave es el dispositivo, los gradientes y los pesos actualizados.

In [6]:
with tf.device("/GPU:0"):
    evaluation = model.evaluate(batches(x_test, y_test), verbose=0, return_dict=True)
assert all(np.isfinite(value) for value in evaluation.values())
report = {
    "tensorflow": tf.__version__, "keras": keras.__version__,
    "python": platform.python_version(), "gpus": gpu_info,
    "matmul_device": product.device,
    "gradient_devices": gradient_devices, "variable_devices": variable_devices,
    "gpu_training_verified": True,
    "training_seconds": training_seconds, "epochs": 2,
    "mnist_train_size": 5400, "mnist_validation_size": 600,
    "mnist_test_size": len(x_test), "evaluation": evaluation,
}
display(evaluation)
(REPORT_DIR / "result.json").write_text(
    json.dumps(report, indent=2, default=str), encoding="utf-8")
print("Evidencia guardada en:", REPORT_DIR / "result.json")


{'accuracy': 0.8960000276565552, 'loss': 0.37874191999435425}

Evidencia guardada en: /home/maxkaizo/hotel_review/artifacts/gpu_validation/result.json


### 7 · Comprobar LSTM nativa y cuDNN por separado

Esta prueba adicional usa secuencias y etiquetas sintéticas pequeñas, no reseñas ni Word2Vec. Ejecuta una actualización en GPU con pre nativo, post nativo y post cuDNN exigido. Si falla cuDNN, el entrenamiento MNIST puede seguir siendo válido: son capacidades diferentes. Registramos cada resultado y los errores, sin sustituir cuDNN por otra implementación.

In [7]:
from tensorflow.keras.utils import pad_sequences
synthetic_sequences = [[2, 3, 4], [5, 6], [7, 8, 9, 10], [2, 8]]
synthetic_labels = np.array([1, 0, 1, 0], dtype="float32")
synthetic_embedding = np.random.default_rng(SEED).normal(0, 0.1, (16, 8)).astype("float32")
synthetic_embedding[0] = 0
lstm_checks = []
for name, padding, cudnn in [
    ("pre_native", "pre", False),
    ("post_native", "post", False),
    ("post_cudnn", "post", True),
]:
    tiny_model = None
    try:
        X = pad_sequences(synthetic_sequences, maxlen=6, padding=padding, dtype="int32")
        with tf.device("/GPU:0"):
            tiny_model = keras.Sequential([
                keras.Input(shape=(6,), dtype="int32"),
                keras.layers.Embedding(
                    16, 8, trainable=False, mask_zero=True,
                    embeddings_initializer=keras.initializers.Constant(synthetic_embedding)),
                keras.layers.LSTM(8, dropout=0.2, use_cudnn=cudnn),
                keras.layers.Dense(1, activation="sigmoid"),
            ])
            tiny_model.compile(optimizer="adam", loss="binary_crossentropy", jit_compile=False)
            tiny_before = [weight.numpy().copy() for weight in tiny_model.trainable_variables]
            tiny_loss = float(tiny_model.train_on_batch(X, synthetic_labels))
            devices = [weight.value.device for weight in tiny_model.trainable_variables]
        assert all("device:GPU:0" in device for device in devices)
        assert np.isfinite(tiny_loss)
        assert any(not np.array_equal(before, after.numpy())
                   for before, after in zip(tiny_before, tiny_model.trainable_variables))
        lstm_checks.append({"case": name, "status": "ok", "loss": tiny_loss, "devices": devices})
    except Exception as error:
        lstm_checks.append({"case": name, "status": "failed",
                            "error": f"{type(error).__name__}: {error}"})
    finally:
        del tiny_model
    print(lstm_checks[-1])
report["lstm_checks"] = lstm_checks
(REPORT_DIR / "result.json").write_text(
    json.dumps(report, indent=2, default=str), encoding="utf-8")


{'case': 'pre_native', 'status': 'ok', 'loss': 0.704708456993103, 'devices': ['/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0']}
{'case': 'post_native', 'status': 'ok', 'loss': 0.6880406141281128, 'devices': ['/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0']}
{'case': 'post_cudnn', 'status': 'ok', 'loss': 0.6940439939498901, 'devices': ['/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:0']}


2249

### Cómo interpretar el resultado

- Si fallan los pasos 1 o 2, todavía no se ha validado el acceso de TensorFlow a GPU.
- Si se completan los pasos 4–6, se verificó entrenamiento del modelo MNIST con gradientes y pesos en GPU.
- El paso 7 comprueba por separado las rutas necesarias para el anexo de la tarea; un error queda visible y registrado.
- Una LSTM nativa en GPU utiliza las operaciones generales de TensorFlow. cuDNN ofrece una implementación especializada; no es el controlador.
- Estas pruebas pequeñas no garantizan memoria suficiente ni rendimiento para Google News y la LSTM de 300 unidades. Eso se verificará en el experimento principal.

Los resultados se escriben en `artifacts/gpu_validation/result.json`. Esta libreta no crea variantes de Colab y no cambia el modelo de reseñas.